## nb_01_test_game_silver

Unit tests for the cleaning/validation functions defined in notebook,
using Python's built-in `unittest` framework and a local Spark session.

Passing `RUN_PIPELINE = False` as a run parameter means only the imports, config,
and function *definitions* from that notebook execute — the `## Run the pipeline`
cells (which read `bronze.game` and write `silver.game`) are skipped entirely, so
this test notebook never touches real bronze/silver tables.

### Load functions from notebook under test (pipeline execution skipped)

In [1]:
%run nb_03_2_game_silver { "RUN_PIPELINE": false }

StatementMeta(, 213ffa05-0120-411f-9a3f-ea653d389a21, 21, Finished, Available, Finished, True)

### Imports

In [2]:
import unittest
from datetime import date
from unittest.mock import MagicMock

from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType
)

StatementMeta(, 213ffa05-0120-411f-9a3f-ea653d389a21, 22, Finished, Available, Finished, False)

### Test data helpers

Small helper to build a `bronze.game`-shaped DataFrame from plain Python rows,
using the current `spark` session provided by the Fabric/Synapse runtime.

In [3]:
RAW_SCHEMA = StructType([
    StructField("game_id", StringType(), True),
    StructField("season", StringType(), True),
    StructField("type", StringType(), True),
    StructField("date_time_GMT", StringType(), True),
])


def make_df(rows: list[tuple]):
    """Build a bronze.game-shaped DataFrame from (game_id, season, type, date_time_GMT) tuples."""
    return spark.createDataFrame(rows, schema=RAW_SCHEMA)

StatementMeta(, 213ffa05-0120-411f-9a3f-ea653d389a21, 23, Finished, Available, Finished, False)

### TestValidateGameDateToSeason

In [4]:
class TestValidateGameDateToSeason(unittest.TestCase):
    """validate_game_date_to_season should:
    1. raise if date_time_GMT is NULL/unparseable
    2. raise if date_time_GMT falls outside the season window
    3. cast valid date_time_GMT to a date-only column
    """

    def test_keeps_valid_in_season_dates(self):
        df = make_df([
            ("1", "20162017", "R", "2016-10-14T23:00:00Z"),  # valid
        ])

        result = validate_game_date_to_season(df)
        self.assertEqual(result.count(), 1)

    def test_raises_on_unparseable_timestamp(self):
        df = make_df([
            ("1", "20162017", "R", None),  # NULL timestamp
        ])
        with self.assertRaises(RuntimeError) as cm:
            validate_game_date_to_season(df)
        self.assertIn("unparseable", str(cm.exception))

    def test_raises_on_date_before_season_start(self):
        df = make_df([
            ("1", "20162017", "R", "2016-08-01T00:00:00Z"),  # before start
        ])
        with self.assertRaises(RuntimeError) as cm:
            validate_game_date_to_season(df)
        self.assertIn("outside", str(cm.exception))

    def test_raises_on_date_after_season_end(self):
        df = make_df([
            ("1", "20162017", "R", "2017-10-15T00:00:00Z"),  # after end
        ])
        with self.assertRaises(RuntimeError) as cm:
            validate_game_date_to_season(df)
        self.assertIn("outside", str(cm.exception))


StatementMeta(, 213ffa05-0120-411f-9a3f-ea653d389a21, 24, Finished, Available, Finished, False)

### Run all tests

In [5]:
loader = unittest.TestLoader()
suite = unittest.TestSuite()
for test_case in (
    TestValidateGameDateToSeason,
):
    suite.addTests(loader.loadTestsFromTestCase(test_case))

runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

print("\n========== FULL FAILURE DETAILS ==========")
print(f"Failures: {len(result.failures)}")
print(f"Errors:   {len(result.errors)}")

for test, traceback in result.failures + result.errors:
    print("\n" + "="*60)
    print(test.id())
    print("="*60)
    print(traceback)


StatementMeta(, 213ffa05-0120-411f-9a3f-ea653d389a21, 25, Finished, Available, Finished, False)

test_keeps_valid_in_season_dates (__main__.TestValidateGameDateToSeason.test_keeps_valid_in_season_dates) ... ok
test_raises_on_date_after_season_end (__main__.TestValidateGameDateToSeason.test_raises_on_date_after_season_end) ... ok
test_raises_on_date_before_season_start (__main__.TestValidateGameDateToSeason.test_raises_on_date_before_season_start) ... 


========== FULL FAILURE DETAILS ==========
Failures: 0
Errors:   0
